# Data

In [277]:
import nltk
from nltk.corpus import gutenberg, stopwords
from nltk import tokenize
import autocorrect
from nltk.stem import WordNetLemmatizer
from autocorrect import Speller
nltk.download('omw-1.4')
import regex
from gensim.models import Word2Vec
import gensim.downloader as api
import pandas as pd


[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Instructions
Objective:
To train word embeddings on famous works of Shakespeare and evaluate their understanding.

Data:
The entire text of plays: 1) The Tragedy of Hamlet, Prince of Denmark, 2) The Tragedy of Macbeth, and 3) The Tragedy of Julius Caesar. These are available from the Gutenberg corpus of the NLTK library. Characters and synopses can be found on Wikipedia.

Problem Statement:
Natural language processing is an important part of the most advanced artificial intelligence software we have today. By studying volumes of text, word embeddings are able to elicit meaning from the words within training data. Your goal is to train a word embedding on three famous works of Shakespeare to determine how well your embedding can understand the meaning of character names and other Shakespearean English words found in these plays.

Steps to be completed:
Create a Jupyter notebook and complete the following steps:



# Data
Use nltk.corpus.gutenberg.raw to load the three plays listed above into a single variable and lower the case.


In [278]:
# Assigning the plays to variables
nltk.download('gutenberg')
hamlet = gutenberg.raw('shakespeare-hamlet.txt')
macbeth = gutenberg.raw('shakespeare-macbeth.txt')
julius_caesar = gutenberg.raw('shakespeare-caesar.txt')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!



Tokenize the text into sentences, and then each sentence into words.


In [279]:
# Keeping all the text in a single variable with normalized case
lower_text = (hamlet + macbeth + julius_caesar).lower()

In [280]:
# Tokenizing sentences
sentences = tokenize.sent_tokenize(lower_text)

In [281]:
# Tokenizing words
words = [tokenize.word_tokenize(sent) for sent in sentences]

Use Speller from the autocorrect library to correct spelling mistakes. 


In [282]:
# Correcting spelling errors
spell = Speller(lang = 'en')
correct_sentences = [[spell(word) for word in sentence] for sentence in words]

Create a list of stopwords (using publicly available lists and/or adding your own) and remove these.


In [283]:
# Removing stop words
stop_words = set(stopwords.words('english'))
no_stop_words = [[word for word in sentence if word not in stop_words] for sentence in correct_sentences]

Use regular expressions (the re library) to do any additional cleanup of the text you wish to do.


Use PorterStemmer or WordNetLemmatizer from nltk.stem on the text.


In [284]:
# Lemmatizing
lemmatizer = WordNetLemmatizer()
lemmatized_words = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in no_stop_words]

In [285]:
# Removing punctuation and then the empty tokens created by the cleanup
regex_cleaned = [[regex.sub(r'[^a-zA-Z0-9]', '', word) for word in sentence] for sentence in lemmatized_words]
regex_cleaned = [[word for word in sentence if word.strip() != ''] for sentence in regex_cleaned]

Print out the words in the first five sentences of the processed text data. (Viewing this may give you additional ideas for the previous steps.)


In [286]:
# Printing the first five sentences
for i in range(5):
    print(f"Sentence {i}: {regex_cleaned[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: ['s']


In [287]:
# Comparing with the actual few sentences
for i in range(5):
    print(f"Sentence {i}: {words[i]}")

Sentence 0: ['[', 'the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', '1599', ']', 'actus', 'primus', '.']
Sentence 1: ['scoena', 'prima', '.']
Sentence 2: ['enter', 'barnardo', 'and', 'francisco', 'two', 'centinels', '.']
Sentence 3: ['barnardo', '.']
Sentence 4: ['who', "'s", 'there', '?']



# Modeling 
Create a CBOW word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data. Use gensim.model.wv.key_to_index  and gensim.model.wv.get_vecattr to print out a list of the 20 most frequent words in the vocabulary along with the word count. Consider improving the text cleaning steps above based on this information. 


# Processing

## Word2Vec models

### CBOW: target word from context words

In [288]:
model = Word2Vec(sentences=regex_cleaned, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [289]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

d                585
ham              337
thou             307
lord             306
shall            300
s                295
come             284
king             248
enter            230
good             221
let              220
mac              205
thy              202
like             200
cesar            193
one              188
make             185
know             184
v                175
thee             174


Based on this word count, we notice a few things:
- d is the top word, which possibly came from words I'd, you'd, we'd. It is just a contraction of 'would', and should probably be removed. Similarly, s and v perhaps correspond to 'is' and 'have'.
- Archaic pronouns like 'thee', 'thy', 'thou' constitute stop words and should also be removed.
- 'Caesar', 'Hamlet' and 'Macbeth' being correct to 'cesar', 'mac', 'ham' sounds lowkey appetizing and defeats the purpose. However, we cannot remove the spelling correction step because the text consists of Latin words which should be converted to English. A better decision might be to manually code those words to preserve the original meaning.

In [290]:
archaic = {"thou", "thee", "thy", "thine", "ye", "hath", "doth"}
stop_words.update(archaic)

In [291]:
regex_cleaned = [[w for w in sent if len(w) > 1] for sent in regex_cleaned]

In [292]:
name_map = {
    "hamlet": "hamlet",
    "ham": "hamlet",
    "macbeth": "macbeth",
    "mac": "macbeth",
    "caesar": "caesar",
    "cesar": "caesar"
}

normalized = [[name_map.get(w, w) for w in sent] for sent in regex_cleaned]

In [293]:
for i in range(5):
    print(f"Sentence {i}: {normalized[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: []


In [294]:
model = Word2Vec(sentences=normalized, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [295]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

hamlet           444
thou             307
lord             306
shall            300
come             284
macbeth          271
king             248
enter            230
good             221
let              220
thy              202
like             200
caesar           193
one              188
make             185
know             184
thee             174
self             166
would            163
aboutus          162


In [296]:
top_5 = ['hamlet', 'cauldron', 'nature', 'spirit', 'general', 'prythee']
for i in top_5:
    print(f'Most similar results for: {i}')
    print(model.wv.most_similar(i, topn=5))


Most similar results for: hamlet
[('polo', 0.9936208724975586), ('good', 0.9919685125350952), ('sweet', 0.9903880953788757), ('faith', 0.989673912525177), ('ratio', 0.9892324805259705)]
Most similar results for: cauldron
[('opinion', 0.9986045360565186), ('burned', 0.9985904097557068), ('tear', 0.9985445737838745), ('horse', 0.9985049366950989), ('kind', 0.9984932541847229)]
Most similar results for: nature
[('whose', 0.9976878762245178), ('even', 0.9976685047149658), ('seems', 0.9974830150604248), ('judgement', 0.997404932975769), ('might', 0.9973195195198059)]
Most similar results for: spirit
[('hide', 0.9980984926223755), ('bone', 0.9977700114250183), ('even', 0.9976897239685059), ('lie', 0.9976701140403748), ('vse', 0.9976627230644226)]
Most similar results for: general
[('cocaine', 0.9988521933555603), ('cyber', 0.9987738728523254), ('draw', 0.9987684488296509), ('ear', 0.9987611770629883), ('danger', 0.9987538456916809)]
Most similar results for: prythee
[('ill', 0.99813747406005

Create a skipgram word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data.


In [297]:
model_sg = Word2Vec(sentences=normalized, vector_size=100, window=5, min_count=3, workers=4, sg=1, epochs=20)

In [298]:
top20_sg = list(model_sg.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20_sg:
    count = model_sg.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

hamlet           444
thou             307
lord             306
shall            300
come             284
macbeth          271
king             248
enter            230
good             221
let              220
thy              202
like             200
caesar           193
one              188
make             185
know             184
thee             174
self             166
would            163
aboutus          162


In [299]:
top_5 = ['hamlet', 'thou', 'lord', 'shall', 'come']
for i in top_5:
    print(f'Most similar results for: {i}')
    print(model_sg.wv.most_similar(i, topn=5))

Most similar results for: hamlet
[('polo', 0.8692903518676758), ('gertrude', 0.86780846118927), ('alert', 0.859654426574707), ('ophelia', 0.8540863990783691), ('queen', 0.8459140658378601)]
Most similar results for: thou
[('wouldst', 0.7852107882499695), ('art', 0.784937858581543), ('didst', 0.7777139544487), ('shalt', 0.7700579166412354), ('stratum', 0.7521200180053711)]
Most similar results for: lord
[('polo', 0.8477759957313538), ('hamlet', 0.8321294784545898), ('good', 0.7837520241737366), ('ala', 0.7716453671455383), ('were', 0.7706266641616821)]
Most similar results for: shall
[('senate', 0.8204192519187927), ('elder', 0.8182931542396545), ('receive', 0.8131200671195984), ('permission', 0.8073581457138062), ('please', 0.8043413162231445)]
Most similar results for: come
[('sit', 0.8422883749008179), ('fetch', 0.8174631595611572), ('field', 0.8035558462142944), ('either', 0.7954208850860596), ('toot', 0.7925920486450195)]


Load the pretrained GloVe model from gensim.models.keyedvectors for comparison with the models trained on Shakespeare text. Use markdown to make note of the data that GloVe has been trained on.

GloVe Training Data
GloVe models available via gensim.downloader were trained on massive general-domain corpora like Wikipedia 2014 + Gigaword 5 (6B tokens, 400K vocab for the 100d version).

In [300]:
# Recommended: glove-wiki-gigaword-100 (matches your vector_size=100)
glove_model = api.load("glove-wiki-gigaword-100")


In [301]:

# Now compare, e.g., on "oeuvre"
print("Skip-gram (Shakespeare):", model_sg.wv.most_similar("hamlet", topn=5))
print("GloVe:", glove_model.most_similar("hamlet", topn=5))

Skip-gram (Shakespeare): [('polo', 0.8692903518676758), ('gertrude', 0.86780846118927), ('alert', 0.859654426574707), ('ophelia', 0.8540863990783691), ('queen', 0.8459140658378601)]
GloVe: [('village', 0.6998987197875977), ('town', 0.6558532118797302), ('situated', 0.5926076769828796), ('located', 0.5660547614097595), ('unincorporated', 0.5599358677864075)]


# Discussion
Compare the three models by finding the 5 most similar terms to each of the following terms: 'hamlet', 'cauldron', 'nature', 'spirit', 'general', and 'prythee'. Comment on how well each model captured the meaning of the word, and if there are multiple meanings, which meaning was given.


In [315]:
terms = ['hamlet', 'cauldron', 'nature', 'spirit', 'general', 'prythee']

# Dictionary to collect ALL results
results_dict = {
    'CBOW': {},
    'Skip-gram': {},
    'GloVe': {}
}

# CBOW
print("=== CBOW ===")
for i in terms:
    try:
        top5 = model.wv.most_similar(i, topn=5)
        print(f'{i}: {top5}')
        results_dict['CBOW'][i] = top5  # Store list of tuples
    except KeyError:
        print(f'{i}: KeyError')
        results_dict['CBOW'][i] = 'KeyError'

# Skip-gram  
print("\n=== Skip-gram ===")
for i in terms:
    try:
        top5 = model_sg.wv.most_similar(i, topn=5)
        print(f'{i}: {top5}')
        results_dict['Skip-gram'][i] = top5
    except KeyError:
        print(f'{i}: KeyError')
        results_dict['Skip-gram'][i] = 'KeyError'

# GloVe
print("\n=== GloVe ===")
for i in terms:
    try:
        top5 = glove_model.most_similar(i, topn=5)
        print(f'{i}: {top5}')
        results_dict['GloVe'][i] = top5
    except KeyError:
        print(f'{i}: KeyError')
        results_dict['GloVe'][i] = 'KeyError'



=== CBOW ===
hamlet: [('polo', 0.9936208724975586), ('good', 0.9919685125350952), ('sweet', 0.9903880953788757), ('faith', 0.989673912525177), ('ratio', 0.9892324805259705)]
cauldron: [('opinion', 0.9986045360565186), ('burned', 0.9985904097557068), ('tear', 0.9985445737838745), ('horse', 0.9985049366950989), ('kind', 0.9984932541847229)]
nature: [('whose', 0.9976878762245178), ('even', 0.9976685047149658), ('seems', 0.9974830150604248), ('judgement', 0.997404932975769), ('might', 0.9973195195198059)]
spirit: [('hide', 0.9980984926223755), ('bone', 0.9977700114250183), ('even', 0.9976897239685059), ('lie', 0.9976701140403748), ('vse', 0.9976627230644226)]
general: [('cocaine', 0.9988521933555603), ('cyber', 0.9987738728523254), ('draw', 0.9987684488296509), ('ear', 0.9987611770629883), ('danger', 0.9987538456916809)]
prythee: [('ill', 0.9981374740600586), ('farewell', 0.9980523586273193), ('marry', 0.9980490803718567), ('wonder', 0.998041570186615), ('therefore', 0.997999370098114)]

=

Compare the three models by finding the cosine similarity between the following pairs of terms: ('brutus', 'murder'), ('lady macbeth', 'queen gertrude'), ('fortinbras', 'norway'), ('rome', 'norway'), ('ghost', 'spirit'), ('macbeth', 'hamlet'). Comment on how well each model captured the similarity between these terms, especially considering the data that each was trained on.


In [319]:
cosine_tests = [
    ['brutus', 'murder'],
    ['lady macbeth', 'queen gertrude'],
    ['fortinbras', 'norway'],
    ['rome', 'norway'],
    ['ghost', 'hamlet'],
]

models = [model, model_sg, glove_model]

for i in cosine_tests:
    w1, w2 = i
    print(f'\nCosine similarity test for: {w1} & {w2}')
    for m in models:
        if m == glove_model:
            try:
                print('\n glove model similarity:', m.similarity(w1, w2))
            except KeyError:
                print('KeyError')

        elif m == model:
            try:
                print('\n CBOW:', m.wv.similarity(w1, w2))
            except KeyError:
                print('KeyError')
            
        elif m == model_sg:
            try:
                print('\n Skip-gram:', m.wv.similarity(w1, w2))
            except KeyError:
                print('KeyError')

            


Cosine similarity test for: brutus & murder

 glove model similarity: 0.07364358
KeyError

 glove model similarity: 0.07364358

Cosine similarity test for: lady macbeth & queen gertrude
KeyError
KeyError
KeyError

Cosine similarity test for: fortinbras & norway

 glove model similarity: -0.028961957

 Skip-gram: 0.929049

 glove model similarity: -0.028961957

Cosine similarity test for: rome & norway

 glove model similarity: 0.28583667

 Skip-gram: 0.4711756

 glove model similarity: 0.28583667

Cosine similarity test for: ghost & hamlet

 glove model similarity: 0.52421004

 Skip-gram: 0.7686179

 glove model similarity: 0.52421004


Compare the three models by finding the 5 most similar terms to each of the following word vectors obtained via linear combination: 'denmark' + 'queen', 'scotland' + 'army' + 'general', 'father' - 'man' + 'woman', 'mother' - 'woman' + 'man'. Comment on how well each model described the ideas behind these word vectors.


In [317]:
analogies = {
    'lc1': {
        "'denmark' + 'queen''": {
            "positive": ['denmark', 'queen'] 
        }
    },
    'lc2': {
        "'scotland' + 'army' + 'general'": {
            'positive': ['scotland', 'army', 'general']
        }
    },
    'lc3': {
        "'father' - 'man' + 'woman'": {
            'positive': ['father', 'woman'],
            'negative': ['man']
        }
    },
    'lc4': {
        "'mother' - 'woman' + 'man'": {
            'positive':['mother', 'man'],
            'negative':['woman']
        }
    }
}

for model in models:

    if model is glove_model:
        print('\n', '=' * 5, 'GloVe (6B)', '=' * 5)
        for lc_id, combos in analogies.items():
            for desc, params in combos.items():
                print(f"\n{desc}:")
                try:
                    top5 = glove_model.most_similar(positive=params.get('positive', []), 
                                           negative=params.get('negative', []), 
                                           topn=5)
                    for word, score in top5:
                        print(f"  {word}: {score:.3f}")
                except KeyError as e:
                    print(f"  Missing word: {e}")

    elif model is model_sg: 
        print('\n', '=' * 5, 'Skip-gram (Shakespeare)', '=' * 5)
        for lc_id, combos in analogies.items():
            for desc, params in combos.items():
                print(f"\n{desc}:")
                try:
                    top5 = model_sg.wv.most_similar(positive=params.get('positive', []), 
                                           negative=params.get('negative', []), 
                                           topn=5)
                    for word, score in top5:
                        print(f"  {word}: {score:.3f}")
                except KeyError as e:
                    print(f"  Missing word: {e}")

    elif model is model:
        print('\n', '=' * 5, 'CBOW (Shakespeare)', '=' * 5)
        for lc_id, combos in analogies.items():
            for desc, params in combos.items():
                print(f"\n{desc}:")
                try:
                    top5 = model.wv.most_similar(positive=params.get('positive', []), 
                                           negative=params.get('negative', []), 
                                           topn=5)
                    for word, score in top5:
                        print(f"  {word}: {score:.5f}")

                except KeyError as e:
                    print(f"  Missing word: {e}")




 ===== CBOW (Shakespeare) =====

'denmark' + 'queen'':
  alert: 0.99830
  ophelia: 0.99812
  macduffe: 0.99765
  attendant: 0.99751
  king: 0.99747

'scotland' + 'army' + 'general':
  ear: 0.99919
  yong: 0.99906
  valiant: 0.99906
  liberty: 0.99902
  micro: 0.99899

'father' - 'man' + 'woman':
  lost: 0.98087
  mother: 0.98076
  oh: 0.98038
  knowst: 0.97822
  fellow: 0.97818

'mother' - 'woman' + 'man':
  blood: 0.98919
  passion: 0.98911
  even: 0.98904
  else: 0.98896
  matter: 0.98882

 ===== Skip-gram (Shakespeare) =====

'denmark' + 'queen'':
  qu: 0.934
  poison: 0.917
  alert: 0.904
  rosincrane: 0.897
  killed: 0.892

'scotland' + 'army' + 'general':
  plebeian: 0.971
  slay: 0.968
  bowl: 0.968
  condemn: 0.967
  train: 0.967

'father' - 'man' + 'woman':
  lost: 0.693
  slain: 0.675
  dear: 0.659
  horrible: 0.650
  deer: 0.649

'mother' - 'woman' + 'man':
  league: 0.629
  alone: 0.626
  brother: 0.618
  qu: 0.611
  father: 0.600

 ===== GloVe (6B) =====

'denmark' + 'que

Give overall comments on how each model performs. Describe what data you would use to train a better word embedding model to captures the meaning of Shakespearean English.